# E079 -- CERN: Particle Resonances in Dimuon Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e079_cern_dimuon.ipynb)

**Objective:** Download CMS Open Data dimuon events from CERN, construct the invariant mass spectrum, and identify fundamental particle resonances — the Z boson (~91 GeV) and J/psi meson (~3.1 GeV) — by fitting Breit-Wigner and Gaussian line shapes.

When two muons are produced in a proton-proton collision, they may come from the decay of a heavier particle. The invariant mass of the muon pair reconstructs the parent particle's mass, revealing peaks in the mass spectrum corresponding to known particles.

## Data Source

- **Experiment:** CMS (Compact Muon Solenoid) at the LHC
- **Dataset:** Dimuon events from 2010 Run B
- **URL:** `http://opendata.cern.ch/record/700/files/MuRun2010B.csv`
- **Columns:** Various kinematic variables; last column `M` is invariant mass in GeV
- **License:** CC0 (CERN Open Data Portal)

In [ ]:
import urllib.request
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Download CMS dimuon data
url = "http://opendata.cern.ch/record/700/files/MuRun2010B.csv"
print("Downloading CMS dimuon data from CERN Open Data Portal...")
req = urllib.request.Request(url, headers={'User-Agent': 'ProtoScience/1.0'})
response = urllib.request.urlopen(req, timeout=300)
raw = response.read().decode('utf-8')
lines = raw.strip().split('\n')
print(f"Downloaded {len(lines) - 1} dimuon events")
print(f"Header: {lines[0]}")

In [ ]:
# Parse — the invariant mass M is the last column
header = lines[0].split(',')
print(f"Columns: {header}")

# Find M column
m_col = len(header) - 1  # usually last
for i, h in enumerate(header):
    if h.strip().upper() == 'M':
        m_col = i
        break

print(f"Mass column index: {m_col} ('{header[m_col].strip()}')")

masses = []
for line in lines[1:]:
    parts = line.split(',')
    if len(parts) > m_col:
        try:
            m = float(parts[m_col].strip())
            if m > 0:
                masses.append(m)
        except ValueError:
            continue

masses = np.array(masses)
print(f"\nParsed {len(masses)} events with positive invariant mass")
print(f"Mass range: [{masses.min():.3f}, {masses.max():.1f}] GeV")
print(f"Median mass: {np.median(masses):.2f} GeV")

## Mass Spectrum and Resonance Fitting

The dimuon invariant mass spectrum should show peaks at:
- **J/psi** (~3.097 GeV): charmonium bound state
- **Upsilon** (~9.46 GeV): bottomonium bound state
- **Z boson** (~91.19 GeV): electroweak gauge boson

We fit the Z boson with a Breit-Wigner (natural line shape of a resonance) and the J/psi with a Gaussian (detector resolution dominated).

In [ ]:
# Z boson: Breit-Wigner fit in [60, 120] GeV window
def breit_wigner(m, A, M0, Gamma, C):
    """Relativistic Breit-Wigner + constant background."""
    return A * Gamma / ((m - M0)**2 + (Gamma/2)**2) + C

z_mask = (masses >= 60) & (masses <= 120)
z_masses = masses[z_mask]
z_counts, z_edges = np.histogram(z_masses, bins=120, range=(60, 120))
z_centers = (z_edges[:-1] + z_edges[1:]) / 2
z_widths = z_edges[1:] - z_edges[:-1]

# Fit
try:
    z_popt, z_pcov = curve_fit(breit_wigner, z_centers, z_counts,
                                p0=[z_counts.max() * 5, 91.2, 2.5, 10],
                                maxfev=10000)
    z_perr = np.sqrt(np.diag(z_pcov))
    print("=== Z Boson Breit-Wigner Fit ===")
    print(f"  Mass M_Z = {z_popt[1]:.2f} +/- {z_perr[1]:.2f} GeV")
    print(f"  Width Gamma_Z = {z_popt[2]:.2f} +/- {z_perr[2]:.2f} GeV")
    print(f"  PDG values: M_Z = 91.1876 GeV, Gamma_Z = 2.4952 GeV")
    z_fit_success = True
except Exception as e:
    print(f"Z fit failed: {e}")
    z_fit_success = False

In [ ]:
# J/psi: Gaussian fit in [2.8, 3.4] GeV window
def gaussian_bg(m, A, mu, sigma, a, b):
    """Gaussian peak + linear background."""
    return A * np.exp(-0.5 * ((m - mu) / sigma)**2) + a * m + b

jpsi_mask = (masses >= 2.8) & (masses <= 3.4)
jpsi_masses = masses[jpsi_mask]
jpsi_counts, jpsi_edges = np.histogram(jpsi_masses, bins=60, range=(2.8, 3.4))
jpsi_centers = (jpsi_edges[:-1] + jpsi_edges[1:]) / 2

try:
    jpsi_popt, jpsi_pcov = curve_fit(gaussian_bg, jpsi_centers, jpsi_counts,
                                      p0=[jpsi_counts.max(), 3.097, 0.05, -100, 500],
                                      maxfev=10000)
    jpsi_perr = np.sqrt(np.diag(jpsi_pcov))
    print("\n=== J/psi Gaussian Fit ===")
    print(f"  Mass M_J/psi = {jpsi_popt[1]*1000:.1f} +/- {jpsi_perr[1]*1000:.1f} MeV")
    print(f"  Resolution sigma = {jpsi_popt[2]*1000:.1f} MeV")
    print(f"  PDG value: M_J/psi = 3096.9 MeV")
    jpsi_fit_success = True
except Exception as e:
    print(f"J/psi fit failed: {e}")
    jpsi_fit_success = False

# Count events in each peak
print(f"\nEvents in Z window [60-120 GeV]: {z_mask.sum()}")
print(f"Events in J/psi window [2.8-3.4 GeV]: {jpsi_mask.sum()}")

In [ ]:
# === 4-SUBPLOT FIGURE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle("E079: CERN Particle Resonances in Dimuon Data (CMS 2010)",
             fontsize=15, fontweight='bold')

# (a) Full mass spectrum (log scale)
ax = axes[0, 0]
ax.hist(masses, bins=300, range=(0.2, 150), color='steelblue',
        edgecolor='none', alpha=0.8)
ax.set_yscale('log')
ax.set_xlabel('Invariant Mass M$_{\\mu\\mu}$ [GeV]', fontsize=11)
ax.set_ylabel('Events / bin', fontsize=11)
ax.set_title(f'(a) Full Dimuon Mass Spectrum (N={len(masses)})', fontsize=12)

# Annotate known resonances
resonances = [(3.097, 'J/ψ'), (9.46, 'Υ'), (91.19, 'Z')]
for m_res, name in resonances:
    ax.axvline(m_res, color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax.text(m_res * 1.05, ax.get_ylim()[1] * 0.3, name,
            fontsize=11, fontweight='bold', color='red')

# (b) Z boson peak with Breit-Wigner fit
ax = axes[0, 1]
ax.bar(z_centers, z_counts, width=z_widths, color='steelblue', alpha=0.7,
       edgecolor='black', linewidth=0.3)
if z_fit_success:
    m_fine = np.linspace(60, 120, 500)
    ax.plot(m_fine, breit_wigner(m_fine, *z_popt), 'r-', linewidth=2.5,
            label=f'Breit-Wigner\nM={z_popt[1]:.2f} GeV\nΓ={z_popt[2]:.2f} GeV')
    ax.axvline(91.1876, color='green', linestyle=':', linewidth=1.5, label='PDG: 91.19 GeV')
ax.set_xlabel('M$_{\\mu\\mu}$ [GeV]', fontsize=11)
ax.set_ylabel('Events / 0.5 GeV', fontsize=11)
ax.set_title('(b) Z Boson Peak', fontsize=12)
ax.legend(fontsize=9)

# (c) J/psi peak with Gaussian fit
ax = axes[1, 0]
jpsi_widths = jpsi_edges[1:] - jpsi_edges[:-1]
ax.bar(jpsi_centers, jpsi_counts, width=jpsi_widths, color='coral', alpha=0.7,
       edgecolor='black', linewidth=0.3)
if jpsi_fit_success:
    m_fine = np.linspace(2.8, 3.4, 500)
    ax.plot(m_fine, gaussian_bg(m_fine, *jpsi_popt), 'r-', linewidth=2.5,
            label=f'Gaussian\nM={jpsi_popt[1]*1000:.0f} MeV\nσ={jpsi_popt[2]*1000:.0f} MeV')
    ax.axvline(3.0969, color='green', linestyle=':', linewidth=1.5, label='PDG: 3097 MeV')
ax.set_xlabel('M$_{\\mu\\mu}$ [GeV]', fontsize=11)
ax.set_ylabel('Events / 10 MeV', fontsize=11)
ax.set_title('(c) J/ψ Peak', fontsize=12)
ax.legend(fontsize=9)

# (d) Upsilon region [8.5, 11] GeV
ax = axes[1, 1]
ups_mask = (masses >= 8.5) & (masses <= 11.5)
ups_counts, ups_edges = np.histogram(masses[ups_mask], bins=60, range=(8.5, 11.5))
ups_centers = (ups_edges[:-1] + ups_edges[1:]) / 2
ups_widths = ups_edges[1:] - ups_edges[:-1]
ax.bar(ups_centers, ups_counts, width=ups_widths, color='mediumpurple', alpha=0.7,
       edgecolor='black', linewidth=0.3)
# Mark the three Upsilon states
for m_ups, label in [(9.460, 'Υ(1S)'), (10.023, 'Υ(2S)'), (10.355, 'Υ(3S)')]:
    ax.axvline(m_ups, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.text(m_ups + 0.05, ax.get_ylim()[1] * 0.8 if ups_counts.max() > 0 else 10,
            label, fontsize=9, color='red')
ax.set_xlabel('M$_{\\mu\\mu}$ [GeV]', fontsize=11)
ax.set_ylabel('Events / 50 MeV', fontsize=11)
ax.set_title('(d) Υ (Upsilon) Region', fontsize=12)

plt.tight_layout()
plt.savefig('e079_cern_dimuon.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved: e079_cern_dimuon.png")

## Key Results Summary

| Particle | Measured Mass | PDG Value | Width/Resolution |
|----------|--------------|-----------|------------------|
| Z boson | From BW fit | 91.188 GeV | Gamma ~ 2.50 GeV |
| J/psi | From Gaussian fit | 3096.9 MeV | Detector sigma |
| Upsilon(1S) | Visible peak | 9460 MeV | Detector limited |

**Physical interpretation:** The dimuon invariant mass spectrum is a window into the fundamental particles of the Standard Model. Each peak corresponds to a particle that can decay into a muon-antimuon pair. The Z boson (discovered 1983, Nobel 1984) mediates the weak nuclear force. The J/psi (discovered 1974, Nobel 1976, "November Revolution") confirmed the existence of the charm quark. The Upsilon states are bound states of bottom quarks, and their three visible peaks reflect the quantum mechanical energy levels of the bb-bar system.